# NB_CursorUsage_Gold
**Source:** `AI_Medallion.Silver.CursorUsage`  
**Target:** `AI_Medallion.Gold.FactCursorActive`  
**Grain:** User + Date (one fact row per silver usage event)  
**Load type:** Incremental (Delta merge on `RecordSurrogateKey`)

In [1]:
%run common_bootstrap

StatementMeta(, d6cd324d-e214-46ff-b0ef-416bc03752f3, 6, Finished, Available, Finished, True)

In [2]:
silver = spark.table(TABLE_CURSOR_SILVER)

request = F.lit(1).cast("int")
use_case_multiplier = F.lit(CURSOR_USE_CASE_MULTIPLIER).cast("decimal(18,8)")
automation_poc = F.lit(CURSOR_AUTOMATION_POC).cast("decimal(18,8)")
total_hours_saved_poc = F.lit(CURSOR_TOTAL_HOURS_SAVED_POC).cast("decimal(18,8)")

StatementMeta(, d6cd324d-e214-46ff-b0ef-416bc03752f3, 7, Finished, Available, Finished, False)

In [3]:
fact = silver.select(
    F.col("CursorUsageMonthNo").alias("Month_No"),
    request.alias("Request"),
    F.lit(CURSOR_ASSET_ID).alias("assetId"),
    F.lit(CURSOR_ASSET_NAME).alias("assetName"),
    F.lit(None).cast("string").alias("eventType"),
    F.lit(None).cast("string").alias("id_project"),
    F.lit(None).cast("string").alias("project"),
    F.lit(None).cast("string").alias("account"),
    F.col("OktaUserId").alias("userId"),
    F.col("OktaUserFullName").alias("username"),
    F.col("OktaUserEmployeeNumber").alias("employeeid"),
    F.col("OktaUserEmail").alias("userEmail"),
    F.col("OktaUserFullEmail").alias("userFullEmail"),
    F.col("OktaUserRegion").alias("Division"),
    F.col("OktaUserCountry").alias("userCountry"),
    F.lit(CURSOR_USE_CASE_CODE).alias("useCaseCode"),
    F.lit(CURSOR_USE_CASE_DISPLAY_NAME).alias("useCaseDisplayName"),
    (request.cast("decimal(18,8)") * total_hours_saved_poc * use_case_multiplier).alias(
        "UseCaseHoursSaved"
    ),
    (
        request.cast("decimal(18,8)")
        * (total_hours_saved_poc / automation_poc)
    ).alias("UseCaseTotalHours"),
    F.lit(1).cast("int").alias("UserWithActivity"),
    F.col("CursorUsageDate").alias("LastModification"),
    F.col("HCBusinessUnit").alias("User_UN"),
    F.col("HCJobTitle").alias("JobRoles"),
    F.col("HCOktaUserCountry").alias("Employee_Country"),
    F.col("HCOffice").alias("Employee_Office"),
    F.col("HCTalentSegment").alias("Segment"),
    F.col("HCDivision").alias("Foundation"),
    F.col("HCBusinessUnitCode").alias("Employee_Business_Unit_Code"),
    F.lit(None).cast("string").alias("asset_model"),
    F.lit(None).cast("string").alias("asset_useCase"),
    F.lit(None).cast("string").alias("asset_mySourceType"),
    F.lit(None).cast("string").alias("asset_sourceType"),
    F.lit(None).cast("string").alias("billingClientName"),
    F.lit(None).cast("string").alias("billingCountry"),
    F.lit(None).cast("string").alias("billingRegion"),
    F.lit(None).cast("string").alias("flowid"),
    use_case_multiplier.alias("useCaseMultiplier"),
    automation_poc.alias("automation_poc"),
    total_hours_saved_poc.alias("totalHoursSaved_poc"),
    F.col(FACT_CURSOR_ACTIVE_MERGE_KEY),
)

# Required not-null columns per spec
fact = fact.filter(
    F.col("Month_No").isNotNull()
    & F.col("Request").isNotNull()
    & F.col("assetId").isNotNull()
    & F.col("assetName").isNotNull()
    & F.col("useCaseCode").isNotNull()
    & F.col("useCaseDisplayName").isNotNull()
    & F.col("UseCaseHoursSaved").isNotNull()
    & F.col("UseCaseTotalHours").isNotNull()
    & F.col("UserWithActivity").isNotNull()
    & F.col("LastModification").isNotNull()
    & F.col("useCaseMultiplier").isNotNull()
    & F.col("automation_poc").isNotNull()
    & F.col("totalHoursSaved_poc").isNotNull()
    & F.col(FACT_CURSOR_ACTIVE_MERGE_KEY).isNotNull()
)

StatementMeta(, d6cd324d-e214-46ff-b0ef-416bc03752f3, 8, Finished, Available, Finished, False)

In [4]:
merge_incremental(
    spark,
    fact,
    TABLE_FACT_CURSOR_ACTIVE,
    FACT_CURSOR_ACTIVE_MERGE_KEY,
)

display(fact.limit(10))

StatementMeta(, d6cd324d-e214-46ff-b0ef-416bc03752f3, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0cfe3748-25a3-467c-b077-23a87c3a78b8)